# End-to-end rockburst assessment demo

Rules FSM path (no LLM). Uses shipped models when present. Held-out examples are primary (m10).

In [1]:
import json
from pathlib import Path
from rockburst.config import load_config, resolve_path

cfg = load_config()
root = Path(cfg.get('_root', '.')).resolve()
def rel(p):
    try:
        return str(Path(p).resolve().relative_to(root))
    except Exception:
        return str(p)
print('processed CSV', rel(resolve_path(cfg, 'processed_csv')))
print('models dir', rel(resolve_path(cfg, 'models')))
print('figures', rel(resolve_path(cfg, 'artifacts') / 'figures'))


processed CSV data/processed/rockburst_processed.csv
models dir artifacts/models
figures artifacts/figures


In [ ]:
model_path = resolve_path(cfg, 'models') / 'best_baseline.joblib'
assert model_path.exists(), 'Run: python -m rockburst.cli train'
print('Using shipped model', model_path)

Using shipped model /home/leviathan/Workspace/agentic-rockburst-model/artifacts/models/best_baseline.joblib


In [3]:
snap_path = resolve_path(cfg, 'baselines') / 'headline_snapshot.json'
snap = json.loads(snap_path.read_text())
print(json.dumps({
    'best_model': snap.get('best_model'),
    'nested_unbiased_qwk': snap.get('nested_unbiased_qwk'),
    'production_qwk_pooled': snap.get('production_qwk_pooled'),
    'conformal_method': snap.get('conformal_method'),
    'conformal_coverage': snap.get('conformal_coverage'),
}, indent=2))

{
  "best_model": "rf__iterative",
  "nested_unbiased_qwk": 0.6972773091316367,
  "production_qwk_pooled": 0.6823578790281211,
  "conformal_method": "mondrian_split",
  "conformal_coverage": 0.9333333333333332
}


In [4]:
from rockburst.graph.runner import assess_case

ex = resolve_path(cfg, 'examples')
heldouts = sorted(ex.glob('heldout_case_*.json'))[:3]
for p in heldouts:
    blob = json.loads(p.read_text())
    print('=' * 40, p.name)
    for k in ['case_no', 'true_intensity', 'predicted_intensity', 'recommended_action',
              'final_system_confidence', 'abstained', 'example_protocol']:
        print(f'  {k}: {blob.get(k)}')

print('=' * 40, 'production showcase case 67 (in-sample)')
prod = assess_case(cfg, case_no=67, use_llm=False)
for k in ['predicted_intensity', 'recommended_action', 'final_system_confidence', 'delivered_planner']:
    print(f'  {k}: {prod.get(k)}')
print('  decision:', (prod.get('decision') or '')[:180])

======================================== heldout_case_13.json
  case_no: None
  true_intensity: None
  predicted_intensity: None
  recommended_action: accept
  final_system_confidence: 0.999999
  abstained: False
  example_protocol: held-out fold model
======================================== heldout_case_15.json
  case_no: None
  true_intensity: None
  predicted_intensity: None
  recommended_action: engineering_verification
  final_system_confidence: 0.6687773809518929
  abstained: False
  example_protocol: held-out fold model: case was in outer test fold; pipeline fit only on fold train/fit indices
======================================== heldout_case_28.json
  case_no: 28
  true_intensity: light
  predicted_intensity: light
  recommended_action: abstain
  final_system_confidence: 0.49
  abstained: True
  example_protocol: held-out fold model: case was in outer test fold; pipeline fit only on fold train/fit indices; rules FSM
======================================== production showca

In [5]:
fa = ex / 'failure_analysis.md'
print(fa.read_text() if fa.exists() else 'missing failure_analysis.md')

# Held-out failure analysis (C5)

Standalone write-up regenerable from shipped models via `scripts/generate_examples.py` (rules FSM; no LLM).

- **True intensity:** strong (grade 3)
- **Predicted intensity:** none (grade 0)
- **|ŷ−y|:** 3
- **R_final:** 0.6875
- **Fold max-prob:** 0.6621621621621622

## Diagnosis

Held-out severe misclassification (|ŷ−y|=3) at high confidence; point prediction none vs true strong. FSM action=abstain. Inter-agent: {'ml_grade': 0, 'ml_intensity': 'none', 'criteria_consensus_grade': None, 'criteria_intensity': 'unknown', 'criteria_grades': {}, 'ml_vs_criteria': False, 'critic_endorse': False, 'critic_rules_fired': ['ensemble_disagreement', 'large_prediction_set', 'wide_conformal_edge', 'validation_objection'], 'parties': ['critic_vs_fusion_path'], 'has_disagreement': True}.

## Decision text

Predicted intensity: none (grade 0). Conformal set: {none, light, moderate} (grades [0, 1, 2]). System confidence R_final=0.688. Recommended action: abstain. Inter-a

In [6]:
fig_dir = resolve_path(cfg, 'artifacts') / 'figures'
for name in ['F01_system_architecture.png', 'F08_model_comparison.png', 'F12_conformal.png']:
    p = fig_dir / name
    print(name, 'OK' if p.exists() else 'MISSING', p.stat().st_size if p.exists() else 0, 'bytes')

F01_system_architecture.png OK 131744 bytes
F08_model_comparison.png OK 367229 bytes
F12_conformal.png OK 93123 bytes
